In [ ]:
# init
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsSync.main as tsm
import os
import boto3
from dotenv import load_dotenv
import pandas as pd
import copy

def pckgs_reload():
    reload(tgm)
    reload(tph)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
logger = tgl.initiate_logger('logger')
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

214


In [1]:
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

214


# read bucket

In [12]:
task_meta = {
    'scrape':{'dir':Path("data/raw/osm countries queries/")}, 
    'clean':{'dir':Path("data/cleaned/")}, 
    'test_basic':{'dir':Path("data/tests results/osm basic test")},
    'test_first_level':{'dir':Path("data/tests results/osm first level test")}
}
task_responses = {t:s3.list_objects_v2(Bucket=bucket_name, Prefix=task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
[len(res['Contents']) for task,res in task_responses.items()]

[18, 83, 2, 10]

In [13]:
sync_state = copy.deepcopy(task_meta)
for task, res in task_responses.items():
    sync_state[task]['b2_objects'] = [obj['Key']  for obj in res['Contents']]
    sync_state[task]['b2_countries'] = tgm.deleteDuplicates([Path(obj['Key']).parent.name  for obj in res['Contents']])
    sync_state[task]['local_countries'] = [dir.name for dir in (ROOT / sync_state[task]['dir']).glob('*') if dir.is_dir()]
    sync_state[task]['b2_not_in_local'] = tgm.complement(
        sync_state[task]['b2_countries'],
        sync_state[task]['local_countries']
    )
    sync_state[task]['local_not_in_b2'] = tgm.complement(
        sync_state[task]['local_countries'],
        sync_state[task]['b2_countries']
    )
# get len
sync_state_resume = copy.deepcopy(sync_state)
for task,val in sync_state_resume.items():
    for k,v in sync_state_resume[task].items():
        if k in ['dir']: continue
        sync_state_resume[task][k] = len(v)

In [14]:
pd.DataFrame(sync_state_resume).T

,dir,b2_objects,b2_countries,local_countries,b2_not_in_local,local_not_in_b2
scrape,data\raw\osm countries queries,18,15,83,0,68
clean,data\cleaned,83,83,83,0,0
test_basic,data\tests results\osm basic test,2,2,83,0,81
test_first_level,data\tests results\osm first level test,10,8,68,0,60


# download data

In [9]:
to_download_countries = list(process_state.keys())
task = 'test_first_level'

In [11]:
tsm.donwload_country_data_from_bucket(to_download_countries, bucket_name, task_meta[task]['dir'], ROOT / task_meta[task]['dir'], s3, logger)

[INFO] :   * Countries to get data: 214
[INFO] :     * Found in B2: 8, Missing in B2: 206
[INFO] :   * Downloading data for countries: 8
[INFO] :     * Directory: 'data\tests results\osm first level test' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm first level test'
[INFO] :     * (1/8) Country Bolivia files found: 1
[INFO] :       * Skip download of existing file Bolivia_first_level_test_res_0.pkl
[INFO] :     * (2/8) Country BosniaAndHerzegovina files found: 1
[INFO] :       * Skip download of existing file BosniaAndHerzegovina_first_level_test_res_0.pkl
[INFO] :     * (3/8) Country Benin files found: 1
[INFO] :       * Skip download of existing file Benin_first_level_test_res_0.pkl
[INFO] :     * (4/8) Country Armenia files found: 2
[INFO] :       * Skip download of existing file Armenia_first_level_test_res_0.pkl
[INFO] :       * Skip download of existing file Armenia_first_level_test_res_1.pkl
[INFO] :     * (5/8) Count

['Bolivia',
 'BosniaAndHerzegovina',
 'Benin',
 'Armenia',
 'Bahrain',
 'Belize',
 'Botswana',
 'BritishVirginIslands']

# upload data

In [21]:
dirs = [DATA_DIR / 'cleaned' / country for country in sync_state['clean']['local_not_in_b2']]
print(len(dirs))

81


In [27]:
config = {'root':ROOT, 's3':s3, 'logger':logger}
for dir in dirs:
    tsm.upload_dir_files_to_backblaze(dir, config)

[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Afghanistan
[INFO] : Number of files found: 1
[INFO] : Uploaded Afghanistan_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Albania
[INFO] : Number of files found: 1
[INFO] : Uploaded Albania_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Algeria
[INFO] : Number of files found: 1
[INFO] : Uploaded Algeria_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Andorra
[INFO] : Number of files found: 1
[INFO] : Uploaded Andorra_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\adminis